# Dia 04 — Ler e escrever arquivos

**🃏 Cartas:** 4 · 25 · 46 &nbsp;·&nbsp; **📘 Original:** *Day 4. Reading and writing files*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 4 — Os depoimentos

Sexta-feira, 14h. Um pen drive com etiqueta escrita à mão chega da delegacia: **"Depoimentos
MegaOferta — cópia"**. Dentro, cinco arquivos de texto, um por vítima, transcritos pelo escrivão.

A Iara espeta o pen drive, copia a pasta para o computador e devolve o original no envelope.

— Regra número um: **a gente nunca trabalha em cima do original**. Copia, e trabalha na cópia.

Você abre o primeiro. Uma senhora de Governador Valadares, uma air fryer, um PIX de R$ 149,90 e
depois outro, de "frete". O segundo, um rapaz de BH e um PlayStation. Os textos são curtos, mas cada
um tem nome, cidade, um número de telefone, uma chave PIX e o mesmo detalhe: *"ele assinava 'Kau'"*.

— Ler cinco é fácil — diz a Iara. — Agora imagina quinhentos. O que a gente quer é o script abrir
cada arquivo, pegar só o que interessa e montar um resumo. E cada coisa que você descobrir, anota
num caderno com data e hora. Caderno de verdade só cresce, nunca apaga.

Ela aponta para a tela.

— Ah, e outra coisa. Se aparecer `DivinÃ³polis` em algum lugar, a culpa não é do escrivão. É sua.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-04-arquivos")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Rodar os quatro exemplos originais: escrever, ler tudo, ler a lista de linhas e ler linha a linha.
- Entender os modos de abertura `"r"`, `"w"`, `"a"` e `"x"`.
- Usar `with open(..., encoding="utf-8")` e saber por que ele evita dois problemas comuns.
- Ler vários arquivos de uma pasta e contar linhas e palavras.
- Extrair campos (vítima, contato, chave PIX) e gravar um resumo em `saidas/`.

## 🧠 Conceito em 2 minutos

Para o Python, um arquivo de texto é uma sequência de linhas. Você **abre**, **lê ou escreve** e
**fecha**:

```python
with open("notas.txt", "a", encoding="utf-8") as f:   # abre (e fecha sozinho no fim do bloco)
    f.write("primeira nota\n")                         # write não pula linha: o \n é seu
```

| Modo | O que faz | Se o arquivo não existe | Se já existe |
|---|---|---|---|
| `"r"` | ler | erro | lê |
| `"w"` | escrever do zero | cria | **apaga tudo** e escreve |
| `"a"` | acrescentar no fim | cria | mantém e acrescenta |
| `"x"` | criar novo | cria | erro (protege contra sobrescrever) |

Três jeitos de ler: `f.read()` (tudo num texto), `f.readlines()` (lista de linhas, cada uma com
`\n`) e `for linha in f:` (uma por vez, sem carregar o arquivo inteiro na memória).

**Codificação.** Sem `encoding=`, o Python usa o padrão do sistema. No Linux e no macOS costuma ser
UTF-8; no Windows em português, muitas vezes é **cp1252**. Um arquivo gravado num e lido no outro
vira `VÃ­tima`. O remédio é sempre o mesmo: `encoding="utf-8"` ao **gravar e ao ler**.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/write_text.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/original/write_text.py) | Acrescenta `"Results text"` ao `results.txt` (modo `"a"`) |
| [`original/read_file.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/original/read_file.py) | Mostra o arquivo inteiro com `read()` |
| [`original/read_lines_loop.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/original/read_lines_loop.py) | Percorre as linhas numerando cada uma |
| [`original/readlines.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/original/readlines.py) | Pega a lista de linhas com `readlines()` e mostra a segunda |
| [`original/results.txt`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/original/results.txt) | Arquivo de exemplo com 3 linhas (criado aqui para os scripts rodarem de primeira) |

```bash
cd dia-04-arquivos/original
python read_file.py
python read_lines_loop.py
python readlines.py
python write_text.py      # rode e depois rode read_file.py de novo: o que mudou?
```

> 💡 O `write_text.py` não coloca `\n`: depois de rodar, a última linha fica
> `...not foundResults text`. Rode duas vezes e veja grudar de novo. Para desfazer (de dentro de `original/`): `git checkout -- results.txt`.
>
> O autor também lembra do jeito mais simples de salvar a saída de qualquer script, sem uma linha de
> código: `python read_file.py > copia.txt` (o `>` manda o que iria para a tela para um arquivo).

### `original/read_file.py` — Ler um arquivo inteiro com read()

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Ler um arquivo inteiro com read()
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_4/read_file.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Não precisa instalar nada. Rode de dentro da pasta original/ (o results.txt está aqui).
# ──────────────────────────────────────────────────────────────────────────

# 💬 open(nome, "r") abre o arquivo para LEITURA ("r" de read).
# 💬 O caminho é relativo: o Python procura results.txt na pasta onde você está.
results_file = open("results.txt", "r")

# 💬 read() devolve o conteúdo INTEIRO como um único texto.
# 💬 O original não fecha o arquivo; o Python fecha sozinho quando o script termina.
print(results_file.read())

### `original/read_lines_loop.py` — Percorrer as linhas de um arquivo numerando cada uma

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Percorrer as linhas de um arquivo numerando cada uma
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_4/read_lines_loop.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Não precisa instalar nada. Rode de dentro da pasta original/ (o results.txt está aqui).
# ──────────────────────────────────────────────────────────────────────────


# 💬 Contador que começa em 1 (para gente, a primeira linha é a 1, não a 0).
stringNumber = 1

# 💬 "with" abre o arquivo e FECHA sozinho no fim do bloco, mesmo se der erro.
with open("results.txt") as f:
    # 💬 Percorrer o arquivo com for lê uma linha por vez (economiza memória em arquivos grandes).
    for line in f:
        # 💬 str() transforma o número em texto para poder juntar com +.
        # 💬 A linha já termina com "\n" e o print pula outra: por isso sai uma linha em branco entre elas.
        print(str(stringNumber) + ". " + line)
        stringNumber += 1

### `original/readlines.py` — Ler as linhas de um arquivo como lista com readlines()

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Ler as linhas de um arquivo como lista com readlines()
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_4/readlines.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Não precisa instalar nada. Rode de dentro da pasta original/ (o results.txt está aqui).
# ──────────────────────────────────────────────────────────────────────────

f = open("results.txt", "r")

# 💬 readlines() devolve uma LISTA: cada item é uma linha do arquivo (com o "\n" no fim).
stringList=f.readlines()

# 💬 [1] é a SEGUNDA linha (a contagem começa em 0). Se o arquivo tiver só uma
# 💬 linha, dá IndexError. Por isso o results.txt de exemplo já vem com várias.
print(stringList[1])

### `original/write_text.py` — Escrever (acrescentar) texto num arquivo

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Escrever (acrescentar) texto num arquivo
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_4/write_text.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Não precisa instalar nada. Rode de dentro da pasta original/ (o results.txt está aqui).
# ──────────────────────────────────────────────────────────────────────────

result = "Results text"

# 💬 Modo "a" (append): ACRESCENTA no fim do arquivo; se ele não existir, cria.
# 💬 Outros modos: "w" apaga tudo e escreve do zero; "x" só cria se ainda não existir; "r" só lê.
results_file = open("results.txt", "a")

# 💬 write() não pula linha sozinho. Rodando duas vezes, fica "Results textResults text".
results_file.write(result)

# 💬 close() garante que tudo foi gravado no disco.
results_file.close()

## 🇧🇷 Nível 2 — Camada Brasil

| Arquivo | O que faz |
|---|---|
| [`br/01_ler_depoimentos.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/br/01_ler_depoimentos.py) | Lê os 5 depoimentos do caso das três formas e monta uma tabela de linhas/palavras |
| [`br/02_anotacoes.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/br/02_anotacoes.py) | Caderno do analista: modos `"x"`, `"a"` e `"w"`, carimbo de hora e a armadilha do cp1252 |
| [`br/03_extrai_campos.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-04-arquivos/br/03_extrai_campos.py) | **O caso:** extrai vítima, cidade, contato, produto, valor e chave PIX; grava resumo `.txt` e `.tsv` |

```bash
cd dia-04-arquivos
python br/01_ler_depoimentos.py
python br/02_anotacoes.py          # rode duas vezes: o caderno cresce
python br/03_extrai_campos.py
```

Nenhum usa internet: os depoimentos estão em `caso/dados/depoimentos/` (fictícios).

**O que você deve ver no caso:** "Kau" aparece nos **5 de 5** depoimentos, todos com o mesmo WhatsApp
`(31) 98888-0101`. E duas chaves PIX se repetem: `f38b2ffc…` recebeu de DEN-001 e DEN-003, e `e5121482…`
de DEN-002 e DEN-004. Mesmas contas recebendo de vítimas diferentes é o começo do mapa do dinheiro
(no Dia 8 você descobre de quem são).

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_ler_depoimentos.py` — Ler os depoimentos: abrir vários arquivos de texto, contar linhas e palavras.

In [ ]:
%cd -q {PASTA_NB}
"""
Ler os depoimentos: abrir vários arquivos de texto, contar linhas e palavras.

Carta 25 (rank B) — Jutsu dos Depoimentos (parte 1)

O que você aprende:
  • a abrir arquivos com "with open(..., encoding='utf-8')" (fecha sozinho e não estraga acentos);
  • as três formas de ler: read() (tudo), readlines() (lista de linhas) e "for linha in arquivo";
  • a achar vários arquivos de uma vez com glob("*.txt");
  • a contar linhas, palavras e caracteres e a montar uma tabela-resumo.

Os depoimentos ficam em caso/dados/depoimentos/ (fictícios). Este script não usa internet.

Como rodar (dentro da pasta dia-04-arquivos):
    python br/01_ler_depoimentos.py
"""

from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
MOSTRAR_PRIMEIRO_INTEIRO = True  #@param {type:"boolean"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_DEPOIMENTOS = PASTA_DIA.parent / "caso" / "dados" / "depoimentos"

# ── 1. Achar os arquivos ──────────────────────────────────────────────────
# glob("*.txt") devolve todos os arquivos que terminam em .txt; sorted() põe em ordem.
arquivos = sorted(PASTA_DEPOIMENTOS.glob("*.txt"))
print(f"📂 {len(arquivos)} depoimentos em {PASTA_DEPOIMENTOS.relative_to(PASTA_DIA.parent)}\n")

# ── 2. Forma 1: read() — o arquivo inteiro num texto só ───────────────────
if MOSTRAR_PRIMEIRO_INTEIRO and arquivos:
    # "with" abre e FECHA o arquivo no fim do bloco, mesmo se acontecer um erro no meio.
    # encoding="utf-8": os depoimentos têm acentos ("Vítima", "Divinópolis").
    with open(arquivos[0], "r", encoding="utf-8") as arquivo:
        texto = arquivo.read()
    print(f"📄 {arquivos[0].name} (lido com read()):")
    print("   " + texto.strip().replace("\n", "\n   "))
    print()

# ── 3. Forma 2: readlines() — uma lista de linhas ─────────────────────────
with open(arquivos[0], encoding="utf-8") as arquivo:
    linhas = arquivo.readlines()
# Cada item termina com "\n" (a quebra de linha). strip() tira isso e os espaços das pontas.
print(f"🔢 readlines() devolveu {len(linhas)} itens. O primeiro, como o Python vê: {linhas[0]!r}")
print(f"   A linha [1] (a segunda) sem o \\n: {linhas[1].strip()!r}\n")

# ── 4. Forma 3: for linha in arquivo — uma linha por vez, para cada depoimento ──
print(f"{'Arquivo':<12} {'Linhas':>6} {'Vazias':>6} {'Palavras':>8} {'Caracteres':>10}")
print("─" * 48)
total_palavras = 0
for caminho in arquivos:
    n_linhas = 0
    n_vazias = 0
    n_palavras = 0
    n_caracteres = 0
    with open(caminho, encoding="utf-8") as arquivo:
        for linha in arquivo:                   # lê uma linha de cada vez (bom para arquivos enormes)
            n_linhas += 1
            if not linha.strip():               # linha que só tem espaço ou "\n" conta como vazia
                n_vazias += 1
            n_palavras += len(linha.split())    # split() sem nada separa por qualquer espaço
            n_caracteres += len(linha.rstrip("\n"))
    total_palavras += n_palavras
    print(f"{caminho.name:<12} {n_linhas:>6} {n_vazias:>6} {n_palavras:>8} {n_caracteres:>10}")

print("─" * 48)
media = total_palavras / len(arquivos) if arquivos else 0
print(f"📊 Total: {total_palavras} palavras · média de {media:.1f} por depoimento")

# ── 5. Uma busca simples: em quantos depoimentos aparece "Kau"? ───────────
# "in" num texto responde se um pedaço aparece dentro dele.
com_kau = [c.name for c in arquivos if "Kau" in c.read_text(encoding="utf-8")]
print(f"🕵️  'Kau' aparece em {len(com_kau)} de {len(arquivos)} depoimentos: {', '.join(com_kau)}")

### `br/02_anotacoes.py` — Caderno de anotações do analista: acrescentar notas com carimbo de data/hora.

In [ ]:
%cd -q {PASTA_NB}
"""
Caderno de anotações do analista: acrescentar notas com carimbo de data/hora.

Carta 25 (rank B) — Jutsu dos Depoimentos (parte 2)

O que você aprende:
  • a diferença entre os modos "w" (apaga e escreve), "a" (acrescenta) e "x" (só cria se não existir);
  • a escrever várias linhas com write() e writelines() (e por que nenhum dos dois pula linha sozinho);
  • a carimbar cada nota com data/hora de Brasília (ISO 8601 com fuso);
  • a armadilha do Windows: sem encoding="utf-8", o Python usa cp1252 e os acentos viram lixo.

Como rodar (dentro da pasta dia-04-arquivos):
    python br/02_anotacoes.py          # rode duas vezes e veja o caderno crescer
"""

from datetime import datetime, timedelta, timezone
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
NOTA = "DEN-003 e DEN-001 pagaram na mesma chave PIX (f38b2ffc…). Pedir extrato ao banco."  #@param {type:"string"}
ANALISTA = "Analista"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
SAIDAS = PASTA_DIA / "saidas"
SAIDAS.mkdir(exist_ok=True)               # cria a pasta se ela ainda não existir
CADERNO = SAIDAS / "caderno_do_analista.txt"

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:                          # Windows sem tzdata: UTC-3 fixo
    FUSO = timezone(timedelta(hours=-3))

# ── 1. Modo "x": criar o cabeçalho só na PRIMEIRA vez ─────────────────────
# "x" dá erro (FileExistsError) se o arquivo já existe. Usamos isso a nosso favor:
# o cabeçalho é escrito uma única vez, nunca duplicado.
try:
    with open(CADERNO, "x", encoding="utf-8") as caderno:
        caderno.write("CADERNO DO ANALISTA — Operação Vitrine Fantasma\n")
        caderno.write("Formato: data/hora ISO 8601 (Brasília) | analista | nota\n")
        caderno.write("=" * 60 + "\n")
    print(f"🆕 Caderno criado: {CADERNO.relative_to(PASTA_DIA)}")
except FileExistsError:
    print(f"📓 Caderno já existe: {CADERNO.relative_to(PASTA_DIA)} (vamos acrescentar)")

# ── 2. Modo "a": acrescentar a nota de hoje, sem apagar as antigas ────────
agora = datetime.now(FUSO).isoformat(timespec="seconds")       # ex.: 2026-10-12T09:15:02-03:00
with open(CADERNO, "a", encoding="utf-8") as caderno:
    # write() NÃO pula linha: o "\n" no fim é obrigatório, senão as notas grudam.
    caderno.write(f"{agora} | {ANALISTA} | {NOTA}\n")
    # writelines() grava uma lista de textos de uma vez (também sem pular linha sozinho).
    caderno.writelines([
        f"{agora} | {ANALISTA} | Lidos 5 depoimentos; 'Kau' citado em todos.\n",
        f"{agora} | {ANALISTA} | Próximo passo: extrair contatos e chaves PIX (br/03).\n",
    ])
print(f"✍️  3 notas acrescentadas às {agora}")

# ── 3. Reler o caderno ────────────────────────────────────────────────────
with open(CADERNO, encoding="utf-8") as caderno:
    conteudo = caderno.readlines()
notas = [l for l in conteudo if " | " in l and l[:4].isdigit()]   # só as linhas que são notas
print(f"\n📚 O caderno tem {len(notas)} notas. As três últimas:")
for linha in notas[-3:]:
    print("   " + linha.rstrip("\n"))

# ── 4. Modo "w": cuidado, ele APAGA tudo ──────────────────────────────────
rascunho = SAIDAS / "rascunho.txt"
with open(rascunho, "w", encoding="utf-8") as r:
    r.write("primeira versão\n")
with open(rascunho, "w", encoding="utf-8") as r:            # abrir de novo com "w"...
    r.write("segunda versão\n")
print(f"\n⚠️  Modo 'w' duas vezes: o rascunho só tem {rascunho.read_text(encoding='utf-8').strip()!r}.")
print("   Para registro de investigação, use 'a': nada some, tudo fica em ordem cronológica.")

# ── 5. A armadilha do cp1252 (Windows) ────────────────────────────────────
# No Windows em português, sem encoding=, o Python (até a versão 3.14) usa cp1252.
# Simulamos aqui o que acontece quando alguém grava em UTF-8 e outro lê como cp1252.
original = "Vítima de Divinópolis pagou R$ 149,90 — chave PIX"
bytes_utf8 = original.encode("utf-8")                        # como fica gravado no disco
lido_errado = bytes_utf8.decode("cp1252", errors="replace")  # como o Windows "antigo" leria
print("\n🪤 A armadilha da codificação")
print(f"   Gravado em UTF-8 : {original}")
print(f"   Lido como cp1252 : {lido_errado}")
print("   Moral: SEMPRE encoding='utf-8' ao gravar E ao ler.")

### `br/03_extrai_campos.py` — Extrair campos dos depoimentos: vítima, contato, chave PIX e valor, num resumo.

In [ ]:
%cd -q {PASTA_NB}
"""
Extrair campos dos depoimentos: vítima, contato, chave PIX e valor, num resumo.

Carta 25 (rank B) — Jutsu dos Depoimentos (parte 3, aplicação no caso)

O que você aprende:
  • a procurar linhas que começam com um rótulo ("Vítima:", "Contato:") usando startswith();
  • a cortar texto com split() e partition() para pegar só o pedaço que interessa;
  • a guardar o que encontrou em dicionários (um por depoimento) e juntar tudo numa lista;
  • a gravar um resumo em texto (para gente ler) e em TSV (para planilha) em saidas/.

Sem expressões regulares: só métodos de texto. No Dia 11 você faz isso com regex.

Como rodar (dentro da pasta dia-04-arquivos):
    python br/03_extrai_campos.py
"""

from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_DEPOIMENTOS = PASTA_DIA.parent / "caso" / "dados" / "depoimentos"
SAIDAS = PASTA_DIA / "saidas"
SAIDAS.mkdir(exist_ok=True)

fichas = []                                   # uma ficha (dicionário) por depoimento

for caminho in sorted(PASTA_DEPOIMENTOS.glob("*.txt")):
    # Começamos a ficha com o id (o nome do arquivo sem .txt) e campos vazios.
    ficha = {"id": caminho.stem, "data": "", "vitima": "", "cidade": "", "contato": "",
             "produto": "", "valor": "", "chave_pix": "", "whatsapp_vendedor": ""}

    with open(caminho, encoding="utf-8") as arquivo:
        for linha in arquivo:
            linha = linha.strip()

            # Linha 1: "DEPOIMENTO DEN-001 — registrado em 2025-11-30"
            if linha.startswith("DEPOIMENTO"):
                ficha["data"] = linha.split("registrado em")[-1].strip()

            # "Vítima: Nome (CPF ...), Cidade/MG"
            elif linha.startswith("Vítima:"):
                resto = linha[len("Vítima:"):].strip()
                # partition(" (") divide em 3 partes: antes, o separador, depois.
                nome, _, depois = resto.partition(" (")
                ficha["vitima"] = nome
                ficha["cidade"] = depois.split("),")[-1].strip()     # "Divinópolis/MG"
                # O CPF fica de FORA do resumo de propósito: ele não é necessário aqui.

            # "Contato: 37993916148"
            elif linha.startswith("Contato:"):
                ficha["contato"] = linha.split(":", 1)[1].strip()    # maxsplit=1: corta só no 1º ":"

            # "Comprei um Air Fryer 5L por R$ 149,90 no site ..."
            elif linha.startswith("Comprei um"):
                meio = linha[len("Comprei um"):].split(" no site")[0]  # "Air Fryer 5L por R$ 149,90"
                produto, _, valor = meio.partition(" por ")
                ficha["produto"] = produto.strip()
                ficha["valor"] = valor.strip()

            # "O site só aceitava PIX. A chave era f38b...419."
            elif "A chave era" in linha:
                ficha["chave_pix"] = linha.split("A chave era")[1].strip().rstrip(".")

            # "Falei com o vendedor pelo WhatsApp (31) 98888-0101; ele assinava 'Kau'."
            elif "WhatsApp" in linha:
                ficha["whatsapp_vendedor"] = linha.split("WhatsApp")[1].split(";")[0].strip()

    fichas.append(ficha)

# ── Mostrar ───────────────────────────────────────────────────────────────
for f in fichas:
    print(f"📄 {f['id']} ({f['data']}) · {f['vitima']} · {f['cidade']}")
    print(f"   🛒 {f['produto']} por {f['valor']}")
    print(f"   💳 PIX: {f['chave_pix']}")
    print(f"   📱 vendedor: {f['whatsapp_vendedor']}  ·  contato da vítima: {f['contato']}")

# ── Cruzamentos simples ───────────────────────────────────────────────────
# Um dicionário "chave PIX → lista de depoimentos" mostra quem pagou para a mesma conta.
por_chave = {}
for f in fichas:
    por_chave.setdefault(f["chave_pix"], []).append(f["id"])   # setdefault cria a lista na 1ª vez

whatsapps = {f["whatsapp_vendedor"] for f in fichas}            # set: só valores diferentes
print("\n🔗 Cruzamentos")
for numero in sorted(whatsapps):
    quantos = sum(1 for f in fichas if f["whatsapp_vendedor"] == numero)
    print(f"   WhatsApp do vendedor {numero}: citado em {quantos} de {len(fichas)} depoimentos")
for chave, ids in por_chave.items():
    print(f"   Chave {chave[:8]}… recebeu de: {', '.join(ids)}")

# ── Gravar o resumo ───────────────────────────────────────────────────────
# 1) Texto para gente ler.
with open(SAIDAS / "resumo_depoimentos.txt", "w", encoding="utf-8") as saida:
    saida.write("RESUMO DOS DEPOIMENTOS — Operação Vitrine Fantasma\n")
    saida.write("(CPFs omitidos de propósito: minimização de dados)\n\n")
    for f in fichas:
        saida.write(f"{f['id']} | {f['data']} | {f['vitima']} | {f['cidade']} | {f['produto']} | "
                    f"{f['valor']} | PIX {f['chave_pix']} | vendedor {f['whatsapp_vendedor']}\n")

# 2) TSV (separado por TAB) para colar numa planilha. utf-8-sig ajuda o Excel a ler os acentos.
colunas = list(fichas[0].keys())
with open(SAIDAS / "resumo_depoimentos.tsv", "w", encoding="utf-8-sig") as saida:
    saida.write("\t".join(colunas) + "\n")
    for f in fichas:
        saida.write("\t".join(f[c] for c in colunas) + "\n")

print("\n💾 Gravados: saidas/resumo_depoimentos.txt e saidas/resumo_depoimentos.tsv")

## ⚖️ Ética, LGPD e OPSEC

- **Trabalhe na cópia, nunca no original.** Um `"w"` errado apaga um depoimento. Mantenha os originais
  intocados (no Dia 14 você aprende a provar isso com hash SHA-256).
- **Minimização.** O resumo do `br/03` deixa o **CPF de fora** de propósito: para cruzar chaves PIX e
  telefones ele não é necessário (LGPD, art. 6º, III). Pegue só o que a pergunta exige.
- **Depoimento de vítima é dado sensível de contexto.** Nome + cidade + valor perdido expõe a pessoa. Não
  copie para e-mail pessoal, pen drive sem controle ou nuvem pública.
- **Caderno com data e hora, só acrescentando.** Um registro que só cresce (modo `"a"`) mostra a ordem
  em que você descobriu as coisas. Isso protege você e a investigação.
- **Cuidado com `saidas/`.** A pasta está no `.gitignore` deste repositório justamente para resumos não
  irem parar no GitHub por engano. Faça o mesmo nos seus projetos.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Você quer registrar notas de investigação sem nunca apagar as anteriores. Qual modo de open() usar?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) 'r'", "B) 'w'", "C) 'a'", "D) 'x'"]
correta = "C) 'a'"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "\"a\" (append) acrescenta no fim. \"w\" apaga o conteúdo anterior, \"r\" só lê e \"x\" só cria um arquivo novo.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Um arquivo gravado em UTF-8 é aberto no Windows com open('arquivo.txt') (sem encoding). O que pode acontecer?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Nada, o Python sempre usa UTF-8", "B) Os acentos viram coisas como 'DivinÃ³polis', porque o padrão pode ser cp1252", "C) O arquivo é apagado", "D) O Python converte o arquivo para PDF"]
correta = "B) Os acentos viram coisas como 'DivinÃ³polis', porque o padrão pode ser cp1252"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Até o Python 3.14, o padrão do open() depende do sistema; no Windows em português costuma ser cp1252. Sempre informe encoding=\"utf-8\".")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Qual a vantagem de abrir arquivos com 'with open(...) as f:'?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) O arquivo fica mais rápido", "B) O arquivo é fechado automaticamente no fim do bloco, mesmo se der erro", "C) Permite abrir arquivos sem extensão", "D) Criptografa o conteúdo"]
correta = "B) O arquivo é fechado automaticamente no fim do bloco, mesmo se der erro"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O with garante o fechamento do arquivo (e a gravação no disco), mesmo que aconteça uma exceção no meio.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 4: Fichas padronizadas e vínculos

**Carta 46 · Rank S**

## A missão

A Delegada Rita gostou do resumo, mas pediu duas coisas:

> "Quero **uma ficha por depoimento**, sempre no mesmo formato, para anexar ao inquérito. E quero saber,
> sem precisar ler tudo, **quando o mesmo número ou a mesma chave PIX aparece em mais de uma denúncia**."

Escreva um script que:

1. Leia todos os arquivos de `caso/dados/depoimentos/`.
2. Extraia, de cada um: data, vítima, cidade, produto, valor, chave PIX e WhatsApp do vendedor.
   Se um campo não existir, escreva `(não informado)`.
3. Grave `saidas/fichas/<id>.txt` no formato `campo: valor`, uma linha por campo, terminando com
   `fonte: <arquivo de origem>`.
4. Mostre os **vínculos**: WhatsApps e chaves PIX que aparecem em mais de um depoimento.
5. Acrescente uma linha com data/hora de Brasília em `saidas/historico.log` a cada execução.
6. Releia uma das fichas e confira se ela bate com o que foi extraído.

## Critérios de rank S

- [ ] Só biblioteca padrão, `encoding="utf-8"` em todo `open()`
- [ ] Fichas com modo `"w"` (refeitas da fonte), histórico com modo `"a"` (nunca apaga)
- [ ] As regras de extração de cada campo ficam num lugar só (um dicionário de regras)
- [ ] O CPF **não** vai para a ficha
- [ ] Aparecem os vínculos das chaves `f38b2ffc…` e `e5121482…`

## Dica

`next((l for l in linhas if rotulo in l), None)` devolve a primeira linha que contém o rótulo, ou `None`.
E `grupos.setdefault(valor, []).append(id)` agrupa ids pelo mesmo valor sem precisar de `if`.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 4 — Ficha de depoimento padronizada (e o diff).

A Delegada Rita quer cada depoimento num formato fixo, "campo: valor", um por arquivo,
e um aviso automático quando o mesmo número de WhatsApp ou a mesma chave PIX aparece em
mais de um depoimento. Este script:
  1. lê caso/dados/depoimentos/*.txt;
  2. grava saidas/fichas/<id>.txt com os campos padronizados (modo "w": a ficha é recriada);
  3. acrescenta (modo "a") uma linha por execução em saidas/historico.log;
  4. imprime os elementos repetidos entre depoimentos (os "vínculos").
Só stdlib, sem internet.
"""

from datetime import datetime, timedelta, timezone
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_DEPOIMENTOS = PASTA_DIA.parent / "caso" / "dados" / "depoimentos"
SAIDAS = PASTA_DIA / "saidas"
PASTA_FICHAS = SAIDAS / "fichas"
PASTA_FICHAS.mkdir(parents=True, exist_ok=True)       # parents=True cria saidas/ e fichas/ de uma vez

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))

# Para cada campo: o rótulo que procuramos na linha e como cortar o valor.
# Uma função pequena (lambda) por campo deixa a regra de cada um num lugar só.
REGRAS = {
    "data":      ("DEPOIMENTO", lambda l: l.split("registrado em")[-1].strip()),
    "vitima":    ("Vítima:",    lambda l: l.split(":", 1)[1].split(" (")[0].strip()),
    "cidade":    ("Vítima:",    lambda l: l.rsplit("),", 1)[-1].strip()),
    "produto":   ("Comprei um", lambda l: l[len("Comprei um"):].split(" por ")[0].strip()),
    "valor":     ("Comprei um", lambda l: l.split(" por ", 1)[1].split(" no site")[0].strip()),
    "chave_pix": ("A chave era", lambda l: l.split("A chave era")[1].strip().rstrip(".")),
    "whatsapp":  ("WhatsApp",   lambda l: l.split("WhatsApp")[1].split(";")[0].strip()),
}

fichas = {}
for caminho in sorted(PASTA_DEPOIMENTOS.glob("*.txt")):
    with open(caminho, encoding="utf-8") as arquivo:
        linhas = [l.strip() for l in arquivo]
    ficha = {}
    for campo, (rotulo, cortar) in REGRAS.items():
        # next(...) pega a PRIMEIRA linha que contém o rótulo (ou None se nenhuma tiver).
        linha = next((l for l in linhas if rotulo in l), None)
        ficha[campo] = cortar(linha) if linha else "(não informado)"
    fichas[caminho.stem] = ficha

    # Ficha padronizada: "campo: valor", um por linha. Modo "w" porque a ficha é REFEITA
    # do zero a partir da fonte sempre que o script roda (a fonte é o depoimento, não a ficha).
    with open(PASTA_FICHAS / f"{caminho.stem}.txt", "w", encoding="utf-8") as saida:
        saida.write(f"id: {caminho.stem}\n")
        saida.writelines(f"{campo}: {valor}\n" for campo, valor in ficha.items())
        saida.write(f"fonte: caso/dados/depoimentos/{caminho.name}\n")

print(f"🗂️  {len(fichas)} fichas gravadas em saidas/fichas/")

# ── Vínculos: valores que se repetem em mais de um depoimento ─────────────
print("\n🔗 Vínculos entre depoimentos")
for campo in ("whatsapp", "chave_pix"):
    grupos = {}
    for ident, ficha in fichas.items():
        grupos.setdefault(ficha[campo], []).append(ident)
    for valor, ids in grupos.items():
        if len(ids) > 1:
            print(f"   🚩 {campo} {valor} aparece em {len(ids)} depoimentos: {', '.join(ids)}")

# ── Histórico de execuções (modo "a": nunca apaga) ────────────────────────
with open(SAIDAS / "historico.log", "a", encoding="utf-8") as log:
    log.write(f"{datetime.now(FUSO).isoformat(timespec='seconds')}\tfichas={len(fichas)}\n")
execucoes = (SAIDAS / "historico.log").read_text(encoding="utf-8").count("\n")
print(f"\n📜 saidas/historico.log: {execucoes} execução(ões) registradas.")

# ── Conferência: a ficha relida bate com o que extraímos? ─────────────────
primeira = next(iter(fichas))
relida = dict(l.rstrip("\n").split(": ", 1) for l in open(PASTA_FICHAS / f"{primeira}.txt", encoding="utf-8"))
ok = all(relida[c] == v for c, v in fichas[primeira].items())
print(f"✅ Conferência da ficha {primeira}: {'tudo bate' if ok else 'DIVERGÊNCIA!'}")

## 🔎 Inovação OSINT do dia

**`Path.copy()` com metadados (Python 3.14).** Desde o Python 3.14, os caminhos do `pathlib` sabem se
copiar sozinhos: `Path("depoimentos/DEN-001.txt").copy("copia/DEN-001.txt", preserve_metadata=True)`.
Com `preserve_metadata=True`, a cópia mantém a **data de modificação** do original (no Windows os
metadados são sempre preservados). Para quem recebe evidências em pen drive, isso evita perder a
cronologia ao fazer a cópia de trabalho, sem precisar do `shutil.copy2`. Também vieram `copy_into()`,
`move()` e `move_into()`. Biblioteca simples: `pathlib`, nativo (no 3.9–3.13, use `shutil.copy2`, que
faz o mesmo).

## 📚 Para ir além

- Tutorial oficial, leitura e escrita de arquivos: <https://docs.python.org/pt-br/3/tutorial/inputoutput.html#reading-and-writing-files>
- `open()` e os modos: <https://docs.python.org/pt-br/3/library/functions.html#open>
- `pathlib` (inclui `copy()`, novo no 3.14): <https://docs.python.org/3/library/pathlib.html>
- Codificações e Unicode, guia oficial: <https://docs.python.org/pt-br/3/howto/unicode.html>

## ✅ Checklist

- [ ] Rodei os quatro scripts originais e vi o `results.txt` mudar
- [ ] Entendi por que `"w"` é perigoso para registros
- [ ] Rodei o `br/02` duas vezes e vi o caderno crescer
- [ ] Vi com meus olhos o `VÃ­tima` da armadilha do cp1252
- [ ] Descobri quais depoimentos pagaram na mesma chave PIX
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 4](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)